In [1]:
import os
import logging
import time
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:
# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("app.log"),  # Archivo de log
        logging.StreamHandler()            # Consola
    ]
)

logger = logging.getLogger(__name__)

In [9]:
def get_products(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_gecko`,  -- Añadidos los backticks
          (SELECT @prompt AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      d.es_marca_propia AS marca,
      d.nombre_matricula_nivel0 AS matricula0,
      d.nombre_matricula_nivel1 AS matricula1,
      d.txt_composicion AS composicion,
      d.forma,
      d.color,
      d.descripcion_visual,
      d.empaque,
      d.zona_de_aplicacion,
      ML.DISTANCE(
        qe.query_embedding,
        d.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_and_embeddings` as d
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """
    
    # Imprimir la consulta SQL generada para depuración
    #print(query)  # Esto te ayudará a verificar la consulta

    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:
        # Asignación de valores con lógica adicional
        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'

        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')

        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query,
        })
    return products

In [ ]:
#Prueba con productos Cofares

def get_products_cofares(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_gecko`,  -- Añadidos los backticks
          (SELECT @prompt AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      d.es_marca_propia AS marca,
      d.nombre_matricula_nivel0 AS matricula0,
      d.nombre_matricula_nivel1 AS matricula1,
      d.txt_composicion AS composicion,
      d.forma,
      d.color,
      d.descripcion_visual,
      d.empaque,
      d.zona_de_aplicacion,
      ML.DISTANCE(
        qe.query_embedding,
        d.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_and_embeddings` as d
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
      WHERE d.es_marca_propia = TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """
    
    # Imprimir la consulta SQL generada para depuración
    #print(query)  # Esto te ayudará a verificar la consulta

    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:
        # Asignación de valores con lógica adicional
        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'

        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')

        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query,
        })
    return products

In [10]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [98]:
# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [99]:
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)
# Lista para almacenar el historial de mensajes
message_history = []
# Construir el historial de la conversación
#historial_conversacion = "\n".join([f"{message['role']}: {message['content']}" for message in message_history])

In [78]:
def refine_query_with_keywords(message_history):

    try:
        # Crear un prompt para el modelo que refine la consulta
        refinement_prompt = """
        Eres un asistente experto en extracción de palabras clave. A continuación, tienes el historial de una conversación:

        Historial de conversación:
        {}

        Extrae las palabras clave más importantes de la consulta del usuario en base al historial y devuélvelas en un formato de texto claro.
        Formato esperado: palabras clave separadas por comas.
        """.format(
            "\n".join([f"{message['role'].capitalize()}: {message['content']}" for message in message_history])
        )

        # Enviar el prompt al modelo para generar el refinamiento
        refinement_response = chat.send_message(refinement_prompt)

        # Obtener la respuesta generada
        prompt = refinement_response.text.strip()

        logger.info(f"Query refinada generada: {prompt}")
        return prompt

    except Exception as e:
        logger.error(f"Error en refine_query_with_keywords: {str(e)}")
        # En caso de error, devolvemos una consulta vacía o un mensaje genérico
        return "consulta vacía"

2024-11-20 11:52:11,175 - INFO - Query refinada generada: Por favor, proporciona el historial de la conversación para que pueda extraer las palabras clave más importantes.


In [100]:
def generate_response(prompt_user):
    # Agregar el mensaje del usuario al historial
    message_history.append({"role": "user", "content": prompt_user})

    # Prompt principal
    instruction_prompt = f"""
    # Instrucción
    Eres Cofinder, un asistente farmacéutico experto.\
    Tu tarea consiste en responder eficazmente a las consultas de los profesionales de farmacia.\
    Te proporcionamos una lista de productos procedentes de la base de datos y previamente rankeados por relevancia.\
    Primero debes leer atentamente la entrada del usuario,\
    y luego desarrollar una respuesta basada en los Criterios proporcionados en la sección Producto a continuación.\
    
    # Producto
    ## Definición de la herramienta
    Tienes acceso a una lista de productos de una base de datos de productos de farmacia "{tools}"\
    que han sido reordenados para proporcionar la mejor respuesta posible a la consulta de un profesional de farmacia.\
    Las instrucciones para realizar la tarea de respuesta a una pregunta se proporcionan en la consulta del usuario.\
    
    ## Criterios
    - Si la entrada del profesional de farmacia es un saludo, preséntese cordialmente como Cofinder el asistente de búsqueda.\
        Ejemplos de saludos: «hola», “hola”, “¿Qué tal?».\
    - Si es necesario, puede pedir detalles aclaratorios para ajustar la búsqueda a resultados eficientes.\
    - Si la entrada solicita búsquedas no relacionadas con productos de farmacia, aclare que ese no es su propósito como asistente de búsqueda de productos de farmacia.\
        Ejemplos de solicitudes no pertinentes: «Quiero la receta de una lasaña», “Quiero pedir una pizza”, “¿Qué tiempo hace hoy?».\
    - Cuando la entrada sea relevante para activar la búsqueda de productos de farmacia, utiliza "tools" para recibir una lista de productos de farmacia clasificados que ayuden al usuario con su tarea. Acepta la solicitud del usuario y proporciónale la lista de productos sin reescribirla.
    - No sugieras ni añadas productos que no estén en la lista proporcionada por el reranker.

    ### Prompt

        Aquí está la consulta del experto farmacéutico: {prompt_user}
    """

    try:
        # Enviar el mensaje al modelo
        response = chat.send_message(instruction_prompt)
        response_text = response.candidates[0].content.parts[0]

        # Agregar la respuesta del modelo al historial
        message_history.append({"role": "assistant", "content": response_text})

        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Refinar la query utilizando el historial actualizado
                    prompt = refine_query_with_keywords(message_history)

                    # Ejecutar búsqueda de productos
                    products = get_products(prompt)
                    if not products:
                        return {
                            "type": "error",
                            "message": "Lo siento, no encontré productos que coincidan con tu búsqueda."
                        }
                    
                    ranked_products = rerank_products(prompt, products)
                    
                    # Devolver directamente la lista de productos
                    return {
                        "type": "product_search",
                        "message": "He encontrado los siguientes productos:",
                        "products": ranked_products["products"]
                    }

        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response_text
        }
                    
    except Exception as e:
        logger.error(f"Error en generate_response: {str(e)}")
        return {
            "type": "error",
            "message": f"Lo siento, ocurrió un error: {str(e)}"
        }

Here are the search requests:

"crema para rejuvenecer la piel de las manos"
respuesta: 'type': 'product_search', 'message': 'He encontrado los siguientes productos:', 'products': 'codigo_web': '163069', 'nombre': 'XHEKPON MANOS 1 TUBO 40 ml', 'codigo_nacional': '1630698', 'descripcion': 'Con colágeno hidrolizado, lípidos y glicerina mantiene el equilibrio hidrolipídico y mejora la resistencia y flexibilidad de la piel, mantiene el grado de hidratación cutánea que le da a la piel un aspecto aterciopelado y suave.', 'modo_implementacion': 'A demanda.IndicacionesIndicado para hidratar y cuidar la piel de las manos.ContraindicacionesMantener en un lugar fresco y seco. Evitar el contacto con los ojos. No ingerir. Mantener fuera del alcance de los niños. Uso tópico. No utilizar si presenta alergia a alguno de los componentes. Para más información, consultar el prospecto del producto. En caso de duda, consulte con su médico o farmacéutico.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/163069.jpg', 'distance_to_query': 0.8927446430874292, 

“Protector solar para piel sensible".
"Crema hidratante para piel atópica".
"Champú anticaída para mujeres".
"Probióticos para mejorar la digestión".
"Vitaminas para fortalecer el sistema inmunológico".
"Gel calmante para dolores musculares".
"Crema antiedad con ácido hialurónico".
"Enjuague bucal para encías sensibles".
"Jarabe para la tos seca".
"Desodorante sin aluminio para pieles sensibles".

In [11]:
# Ejemplo de uso
prompt_user = "Protector solar para piel sensible"

In [12]:
products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: LADIVAL FACIAL PIEL SENSIBLE CON COLOR FPS 50+ 1 ENVASE 50 ml, Descripción: Gel crema con protección solar muy alta (SPF 50+) para las pieles más sensibles o alérgicas. Su textura permite una rápida absorción y es oil free, por lo que se tolera perfectamente por la piel con tendencia acneica. Protege en profundidad para evitar reacciones de alergia o intolerancia al sol y gracias a su formulación SIN (sin parabenos, sin emulsionantes, sin perfumes, sin conservantes) respeta la piel y evita reacciones alérgicas. Protege frente a las radiaciones UVA, UVB e IR-A; siendo la última la principal responsable del fotoenvejecimiento. Esta fórmula tiene color para un acabado mate, luminoso y uniforme. Es resistente al agua. Indicada para piel grasa, mixta o normal., Modo de implementación: Aplicar diariamente sobre la piel limpia del rostro, cuello escote y manos antes de la exposición al sol. Repetir la aplicación regularmente, en especial después del baño.Indicacio

In [13]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: Eucerin Sun Spray Transparente Sensitive Protect FPS 50, 200 ml, Descripción: Eucerin Sun Spray Transparente Sensitive Protect FPS 50 es un protector solar corporal. Está indicado principalmente para piel sensible y piel con tendencia acneica. Combina filtros solares que protegen de los rayos UVA y UVB, a través del Advanced Spectral Technology. Proporciona una protección alta (FPS 50), que se completa con licocalcón A para neutralizar los radicales libres que generan los rayos del sol. Su formato en spray permite una fácil aplicación y deja un acabado transparente, al absorberse rápidamente. Es resistente al agua y no deja rastro graso. Comprar Eucerin Sun Spray Transparente Sensitive Protect FPS 50 online ahora es más fácil y sencillo., Modo de implementación: ¿Cómo aplicar Eucerin Sun Spray Transparente Sensitive Protect FPS 50 Pulverizar el spray por toda la piel del cuerpo en cantidad suficiente, media hora antes de la exposición solar. Renovar cada 2 

In [113]:
response_text = generate_response(prompt_user)  # Generar la respuesta
print(response_text)

{'type': 'conversation', 'message': text: "Entiendo que estás buscando un protector solar, pero necesito más información para poder ayudarte. \n\n¿Podrías decirme qué tipo de protección buscas? ¿Para niños o adultos? ¿Factor alto, medio o bajo? \n\nCon más detalles, puedo mostrarte los productos más adecuados de nuestra base de datos. \n\n\n"
}
